In [ ]:
import os
import numpy as np
import nibabel as nib
import cv2
from pathlib import Path
from scipy.io import savemat

def sharp_downsize_7t():
    # 1. Configuration
    n_train, n_valid, n_test = 2, 2, 2
    total_needed = n_train + n_valid + n_test
    
    # Path Handling (Modern Python way)
    # Equivalent to strrep(pwd, 'util', 'data')
    current_dir = Path.cwd()
    data_root = Path(str(current_dir).replace('util', 'data'))
    
    # Get subject directories (Equivalent to dir() minus the . and ..)
    # We sort to ensure consistency across different OS
    all_files = sorted([f for f in data_root.iterdir() if f.is_dir()])
    
    min_vals = []
    max_vals = []
    info_headers = []

    # 2. Main Loop
    for i, subject_dir in enumerate(all_files[:total_needed]):
        subject_name = subject_dir.name
        nifti_path = subject_dir / "T1w" / "Diffusion_7T" / "data.nii.gz"
        
        if not nifti_path.exists():
            print(f"Skipping {subject_name}: NIfTI not found at {nifti_path}")
            continue

        # Load NIfTI (nibabel is the standard for .nii.gz)
        img = nib.load(str(nifti_path))
        temp = img.get_fdata()
        
        # Collect Stats
        min_vals.append(np.min(temp))
        max_vals.append(np.max(temp))
        info_headers.append(str(img.header))

        # Permute [2 1 3 4] and mat2gray (Normalize 0-1)
        # MATLAB [2 1 3 4] -> Python (1, 0, 2, 3) because of 0-based indexing
        temp = np.transpose(temp, (1, 0, 2, 3))
        
        t_min, t_max = np.min(temp), np.max(temp)
        if t_max > t_min:
            temp = (temp - t_min) / (t_max - t_min)
            
        # Crop (1:206, 1:172)
        temp = temp[0:206, 0:172, :, :]

        # 3. Determine Split Destination
        if i < n_train:
            dest = 'train'
        elif i < n_train + n_valid:
            dest = 'valid'
        else:
            dest = 'test'

        # 4. Prepare Directories (Handles the NAS write issue)
        # This creates gt, x2, and x3 paths
        paths = {
            'gt': data_root / 'gt' / dest / subject_name,
            'x2': data_root / 'x2' / dest / subject_name,
            'x3': data_root / 'x3' / dest / subject_name
        }
        
        for p in paths.values():
            p.mkdir(parents=True, exist_ok=True)

        # 5. Iterative Resizing and Saving
        # Shape is (H, W, Slices, B-vals)
        _, _, num_slices, num_bvals = temp.shape

        print(f"Processing {subject_name} ({dest})...")
        
        for j in range(num_bvals):
            for k in range(num_slices):
                # Slice data
                slice_data = temp[:, :, k, j]
                
                # Resize using OpenCV (Inverse of NumPy shape)
                h, w = slice_data.shape
                lr_x2 = cv2.resize(slice_data, (w//2, h//2), interpolation=cv2.INTER_CUBIC)
                lr_x3 = cv2.resize(slice_data, (w//3, h//3), interpolation=cv2.INTER_CUBIC)

                # Format filename
                fname = f"{subject_name}-frame{j+1:03d}-slice{k+1:03d}.png"
                
                # Save 16-bit PNG (Equivalent to im2uint16 + imwrite)
                def save_16bit(data_arr, target_path):
                    # Scale to 0-65535 and cast to uint16
                    out = (data_arr * 65535).astype(np.uint16)
                    cv2.imwrite(str(target_path / fname), out)

                save_16bit(slice_data, paths['gt'])
                save_16bit(lr_x2, paths['x2'])
                save_16bit(lr_x3, paths['x3'])

        print(f"Folder {i+1}: {subject_name} done!")

    # 6. Save Metadata to .mat
    savemat('test_7T_16b.mat', {
        'max_val': max_vals,
        'min_val': min_vals,
        'info': info_headers
    })

if __name__ == "__main__":
    sharp_downsize_7t()